# DEX/No-DEX Footprinting

#### Files:

- `Glucocorticoid receptor (GR/NR3C1?) BED file`: List of genomic sites where GR binds [chromosome, start, end, name, score, strand]
    - Generated from JASPAR NR3C1 MA0113.1 MEME file using FIMO
- `Tn5 Bias File for Mice`: genome=scp.genome.mm10; already including in scPrinter
- `Fragment file`: tbd....

In [1]:
import os
import numpy as np
import sys
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
os.environ.pop("SCPRINTER_DATA", None)
import scprinter as scp
import scprinter.tools as _sct
from concurrent.futures import Future
import h5py

#workaround for failing parallelism (see note)
class _SerialExecutor: 
    def __init__(self, *args, **kwargs): pass
    def submit(self, fn, *args, **kwargs):
        f = Future()
        try:
            f.set_result(fn(*args, **kwargs))
        except Exception as e:
            f.set_exception(e)
        return f
    def shutdown(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *exc): self.shutdown()

_sct.ProcessPoolExecutor = _SerialExecutor

import time
import pandas as pd
import pybedtools
import matplotlib.pyplot as plt
import re
import math
import copy
import pyBigWig
from tqdm import tqdm
from zipfile import ZipFile
import seaborn as sns
import scipy.stats as ss
import torch

/Users/cherylxiang/miniforge3/envs/scprinter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wd = os.path.expanduser('/Users/cherylxiang/Documents/GitHub/printing/data')
os.makedirs(os.path.join(wd, "plots"), exist_ok=True)

In [3]:
barcodes = ['no_dex', 'dex']

### Load GR BED File

| Col | Content |
| --- | --- |
| 0 | chromosome |
| 1 | motif start (0-based, inclusive) |
| 2 | motif end (0-based, exclusive) |
| 3 | name + seq |
| 4 | p-value: smaller = stronger match to GR motif |
| 5 | negative/positive strand |

In [4]:
gr_bed = pd.read_table('data/GR_motifs_mm10.bed', header=None)
print('sites (filtered to p-val < 0.0001):', gr_bed.shape[0])
gr_bed.head(10)

sites (filtered to p-val < 0.0001): 519243


,0,1,2,3,4,5
0,chr1,3002020,3002038,GR_GAGAAGAAAACTTCCCAA,0.000015,-
1,chr1,3003351,3003369,GR_GAGAACAGTCTGTGCAGA,0.000070,-
2,chr1,3012791,3012809,GR_AAAAACAGTCAGTTGTGC,0.000061,+
3,chr1,3014633,3014651,GR_AAAACCAGCCTGACCAGG,0.000087,+
4,chr1,3021987,3022005,GR_AGAAGCATACAGAACTGT,0.000072,+
5,chr1,3023377,3023395,GR_GGAAACAAAAAGAACTAT,0.000033,+
6,chr1,3026770,3026788,GR_AGGAAAAAAATTTCCAAT,0.000074,-
7,chr1,3034095,3034113,GR_AAAAACATCATGTTCTAC,0.000001,-
8,chr1,3034271,3034289,GR_AGAAACAAAAAGAACTAT,0.000055,-
9,chr1,3036023,3036041,GR_AAGAAGATTATTTTCTTG,0.000029,+


### Download mm10 Files

In [5]:
scp.genome.mm10.fetch_bias()       # mouse Tn5 bias (.h5)
bw = scp.genome.mm10.fetch_bias_bw()
b = pyBigWig.open(bw); print(b.values("chr1", 3002020, 3002030)); b.close() 

100%|█████████████████████████████████████| 10.9G/10.9G [00:00<00:00, 10.3TB/s]


creating bias bigwig (runs for new bias h5 file)


100%|██████████| 66/66 [24:58<00:00, 22.71s/it]  


bias bigwig created
[0.0876321941614151, 0.43210163712501526, 0.2552497982978821, 0.25499406456947327, 0.038610637187957764, 0.03572392463684082, 0.3674987554550171, 0.06727196276187897, 0.277785986661911, 0.0833393782377243]


In [ ]:
#loads printer_data.h5ead if it has already been imported, otherwise imports the frags
h5ad_path = os.path.join(wd, 'GR_printer.h5a')
done_flag = os.path.join(wd, 'GR_printer.DONE') 

if os.path.exists(done_flag):
    print("Loading existing printer object...")
    printer = scp.load_printer(h5ad_path, scp.genome.mm10)
else:
    print("Importing fragments (first run, this takes a while)...")
    if os.path.exists(h5ad_path):
        os.remove(h5ad_path)            # clear any half-written file from an interrupted run
    printer = scp.pp.import_fragments(
        path_to_frags=os.path.join(wd, 'TBA'),
        barcodes=barcodes,
        savename=h5ad_path,
        genome=scp.genome.mm10,
        sorted_by_barcode=True)
    open(done_flag, 'w').close()        # mark as finished

#loads tn5 dispersion model --> pretrained on deproteinized ATAC-seq data 
printer.load_disp_model()